# Benchmark: Precomputed Keys

Loads pre-computed data from `benchmark_precomputed.csv` (generated by
`run_benchmark_precompute.py`) and produces a three-bar plot comparing:

1. **naive**       — each pair processed independently; both SMILES are
   re-classified on every occurrence (no cross-pair reuse).
2. **standard**    — `enumerate_pairs(parallel=True)` with the package's
   default per-call deduplication; each unique SMILES is classified once
   per call.
3. **precomputed** — `preprocess_smiles` builds the key map up-front and
   `enumerate_pairs(..., precomputed_keys=...)` skips classification
   entirely; the preprocessing cost is reported separately as a one-time
   amortised cost.

Pairs are sampled from a pool of ~100,000 building blocks so that each
SMILES recurs in many pairs (~32× average), the regime in which
deduplication actually pays off.

## Reproducing the data

The CSV consumed by this notebook lives under `data/paper/`, which is **not checked into git**. Regenerate it with:

```bash
pixi run -e dev python scripts/run_benchmark_precompute.py
```


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PRECOMPUTED_FILE = "../../data/paper/benchmark_precomputed.csv"

In [ ]:
df_pre = pd.read_csv(PRECOMPUTED_FILE)
df_pre

In [ ]:
row_naive = df_pre[df_pre["method"] == "naive"].iloc[0]
row_std   = df_pre[df_pre["method"] == "standard"].iloc[0]
row_pre   = df_pre[df_pre["method"] == "precomputed"].iloc[0]

labels = [
    "Naive\n(no deduplication)",
    "Standard\n(per-call dedup)",
    "Precomputed\n(enumerate only)",
]
means  = [row_naive["mean_pps"], row_std["mean_pps"], row_pre["mean_pps"]]
stds   = [row_naive["std_pps"],  row_std["std_pps"],  row_pre["std_pps"]]
colors = ["#F44336", "#2196F3", "#4CAF50"]  # red / blue / green

preprocess_mean = row_pre["preprocess_mean"]
preprocess_std  = row_pre["preprocess_std"]

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    labels, means, yerr=stds, color=colors, alpha=0.85,
    capsize=6, error_kw={"linewidth": 1.5},
)

for bar, mean in zip(bars, means):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        mean + max(stds) * 1.3,
        f"{mean:,.0f} pairs/s",
        ha="center", va="bottom", fontsize=13,
    )

ax.set_ylabel("Pairs / second", fontsize=15)
ax.tick_params(axis="both", labelsize=13)
ax.set_ylim(bottom=0, top=max(means) * 1.18)
ax.grid(True, axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()

out = Path("../../smartreact_paper/figures/precomputed_keys.png")
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=300, bbox_inches="tight")
print(f"Saved {out}")
plt.show()

## Summary

In [ ]:
n_pairs = int(row_naive["n_pairs"])
wall_naive = n_pairs / row_naive["mean_pps"]
wall_std   = n_pairs / row_std["mean_pps"]
wall_pre   = n_pairs / row_pre["mean_pps"]

summary = pd.DataFrame(
    [
        {
            "method": "naive",
            "pairs_per_s": round(row_naive["mean_pps"], 1),
            "wall_s": round(wall_naive, 1),
            "vs_naive": 1.0,
            "vs_standard": round(row_naive["mean_pps"] / row_std["mean_pps"], 2),
        },
        {
            "method": "standard",
            "pairs_per_s": round(row_std["mean_pps"], 1),
            "wall_s": round(wall_std, 1),
            "vs_naive": round(row_std["mean_pps"] / row_naive["mean_pps"], 2),
            "vs_standard": 1.0,
        },
        {
            "method": "precomputed",
            "pairs_per_s": round(row_pre["mean_pps"], 1),
            "wall_s": round(wall_pre, 1),
            "vs_naive": round(row_pre["mean_pps"] / row_naive["mean_pps"], 2),
            "vs_standard": round(row_pre["mean_pps"] / row_std["mean_pps"], 2),
        },
    ]
)
summary

In [ ]:
delta_classify = wall_std - wall_pre
print(f"n_pairs              : {n_pairs:,}")
print(f"naive  \u2192 standard    : {row_std['mean_pps'] / row_naive['mean_pps']:.2f}\u00d7 (per-call dedup)")
print(f"standard \u2192 precomputed: {row_pre['mean_pps'] / row_std['mean_pps']:.2f}\u00d7 (skip classification)")
print(f"naive  \u2192 precomputed : {row_pre['mean_pps'] / row_naive['mean_pps']:.2f}\u00d7 (combined)")
print()
print(f"standard wall - precomputed wall = {delta_classify:.1f} s")
print(f"preprocess one-time cost         = {preprocess_mean:.1f} s")
print("  (the two should agree: precomputed simply moves the per-call\n"
      "   classification work out of the enumeration call.)")